# Huấn luyện detector trên Kaggle
Chạy cell lần lượt, bật Internet/GPU. Chỉnh các cell cấu hình bên dưới ngay tại notebook, không cần sửa settings.py.
Có thể chọn 1, 2 hoặc cả 3 kiến trúc; mỗi kiến trúc chạy các seed bạn chọn.
Sau train: xem báo cáo **riêng từng detector/seed trên validation**, không tự xếp hạng model. Test là bước tùy chọn, tắt mặc định.
Notebook compare.ipynb chỉ là khung dành cho công việc so sánh sau này.
Worker/checkpoint thật vẫn cần kiểm chứng qua pilot Kaggle; kiểm thử phần mềm không thay thế bước này.


In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
assert (ROOT / "settings.py").is_file(), "Cần upload/push code mới trước khi dùng notebook"
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT / "data_pipeline"), "-e", str(ROOT) + "[training]"], check=True)
import settings
settings.bootstrap()
from vn_av_df.actions import execute


## 1. Chọn part, kiến trúc và tên lần chạy
DATASET_PARTS chọn dữ liệu; ARCHITECTURES chọn model. Hai lựa chọn độc lập.
Tên RUN_NAME là folder output, ví dụ train_part1_gru_run01 hoặc train_parts1_2_all_run01.
Đổi dữ liệu/model/hyperparameter: dùng tên run mới. RESUME chỉ dùng khi tiếp tục đúng lần chạy bị ngắt.


In [ ]:
DATASET_ROOT = Path("/kaggle/input")
# Trỏ đến folder có manifest.jsonl + dataset_info.json; sửa đúng mount path Kaggle của bạn.
DATASET_PARTS = ["vn-av-df-data-part1/vn-av-df-data-part1"]
# Một part khác: ["vn-av-df-data-part2/vn-av-df-data-part2"]
# Hai part:
# DATASET_PARTS = ["vn-av-df-data-part1/vn-av-df-data-part1", "vn-av-df-data-part2/vn-av-df-data-part2"]
# DATASET_PARTS = "all"  # Mọi part đã attach, không tự download.
# Nếu tất cả nằm trong một Kaggle dataset:
# DATASET_ROOT = Path("/kaggle/input/YOUR_COLLECTION/vn-av-df-data")
# DATASET_PARTS = ["vn-av-df-data-part1", "vn-av-df-data-part2"]

ARCHITECTURES = "all"  # B-FATE (fate_gru), B-AVH (avh_tcn) và P2 (p2_syncartifact).
# ARCHITECTURES = ["fate_gru"]  # Chỉ B-FATE, không cần worker AV-HuBERT.
# ARCHITECTURES = ["p2_syncartifact"]  # Chỉ P2: AV-HuBERT + DINOv2, không tải FATE.
# Ablation P2 (dùng lại cache AV-HuBERT/DINOv2):
# ARCHITECTURES = ["avh_realrecon", "p2_sync_only", "p2_artifact_only", "p2_concat", "p2_sync_seen_fake"]
RUN_NAME = "train_part1_run01"
DEVICE = "cuda"
SEEDS = [42, 43, 44]  # Pilot: [42]. 3 kiến trúc x 3 seed = 9 lượt train detector.
RESUME = False
RUN_TEST = False  # True chỉ khi đã chốt cấu hình; không dùng test để chọn model.
CACHE_DIR = ROOT / "cache/vn-av-df-data"  # Cache có thể dùng lại nếu cùng media/encoder.
HELD_OUT_GENERATORS = ["musetalk_1_5"]  # Generator chỉ dùng test trong protocol chính.
# Pilot chỉ có Wav2Lip: đặt [] (đây là protocol khác; dùng RUN_NAME riêng).


## 2. Thông số train và trích feature
TRAINING áp dụng cho detector được chọn; RECONSTRUCTION chỉ dùng nhánh có bộ tái dựng A→V.
Linear không dùng hidden/dropout; hiện training xử lý một clip mỗi bước tối ưu, không có batch_size detector.
Khi đổi thông số trích feature, cần cache phù hợp và RUN_NAME mới. Worker chỉ được cài nếu kiến trúc cần nó.


In [ ]:
TRAINING = {
    "epochs": 20,             # Pilot: 2.
    "lr": 3e-4,              # Learning rate AdamW.
    "weight_decay": 1e-4,    # Regularization.
    "hidden": 128,           # Kích thước hidden/projection của temporal head.
    "dropout": 0.1,
    "patience": 5,           # Dừng sau 5 epoch validation loss không cải thiện; 0 = tắt.
    "top_fraction": 0.1,     # Score video = mean top 10% ô hợp lệ.
    "torch_threads": 2,      # Luồng CPU, không phải batch size.
    "balance_parents": True, # Mỗi epoch lấy một variant/parent để cân bằng nguồn.
    "artifact_aux_weight": 0.5,  # P2: trọng số head phụ artifact (nhãn AI theo ô).
}
RECONSTRUCTION = {
    "epochs": 20,            # Pilot: 2. Stage A của P1/P2: chỉ real train/validation.
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "patience": 5,
}
SYNC = {
    "epochs": 20,            # Pilot: 2. Stage S của P2: real, sham, real dịch lệch; không thấy fake.
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "patience": 5,
    "shift_frames": [3, 15],  # Dịch audio ±3–15 frame (0,12–0,6 s) để tạo mẫu lệch.
    "shift_probability": 0.5, # Tỷ lệ real được dịch lệch mỗi epoch.
}
FATE_FEATURES = {
    "precision": "float16",  # CPU: đổi thành float32.
    "window_s": 2.0, "step_s": 0.2, "temporal_bins": 8,
    "min_coverage": 0.75, "max_duration": 60, "max_side": 640,
}
AVH_FEATURES = {
    "chunk_frames": 200, "overlap_frames": 50,
    "output_stride": 5,      # 25 Hz / 5 = output 5 Hz, mỗi ô 0.2 s.
    "max_duration": 60, "max_side": 640,
}
DINO_FEATURES = {
    "precision": "float16",  # CPU: đổi thành float32.
    "crop_size": 224,        # Crop miệng RGB cho DINOv2 ViT-S/14, bội số 14.
    "batch_size": 64,        # Giảm nếu thiếu VRAM.
}
BOOTSTRAP_REPLICATES = 1000  # CI khi chủ động chạy test; không ảnh hưởng train.


In [ ]:
MAIN_METHODS = ["fate_gru", "avh_tcn", "p2_syncartifact"]
chosen = MAIN_METHODS.copy() if ARCHITECTURES == "all" else ARCHITECTURES
if not isinstance(chosen, list) or not chosen or len(set(chosen)) != len(chosen):
    raise ValueError("ARCHITECTURES phải là 'all' hoặc danh sách model không trùng")
settings.DATASET, settings.DATASET_PARTS = DATASET_ROOT, DATASET_PARTS
settings.ARCHITECTURES, settings.SEEDS = chosen, SEEDS
settings.RUN_NAME, settings.DEVICE = RUN_NAME, DEVICE
settings.DEMO_METHOD = chosen[0]  # Chỉ làm mặc định demo, không phải model tốt nhất.
cfg = settings.config()
if set(chosen) - set(cfg["methods"]):
    raise ValueError("Tên model không có trong settings.config()['methods']")
cfg["cache"] = str(CACHE_DIR)
cfg["held_out_generators"] = HELD_OUT_GENERATORS
cfg["training"].update(TRAINING)
cfg["reconstruction"].update(RECONSTRUCTION)
cfg["sync"].update(SYNC)
cfg["encoders"]["fate"].update(FATE_FEATURES)
cfg["encoders"]["avhubert"].update(AVH_FEATURES)
cfg["encoders"]["dinov2"].update(DINO_FEATURES)
cfg["bootstrap_replicates"] = BOOTSTRAP_REPLICATES
from vn_av_df.dataset import training_dataset
rows, selection = training_dataset(cfg, verify_media=True)
print("Part và hash:", selection)
print("Model:", chosen, "| Seeds:", SEEDS, "| Output:", cfg["runs"])
execute("encoder_setup", cfg)  # Chỉ tải encoder của các model đã chọn.


In [ ]:
needs_avhubert = any(cfg["methods"][name]["encoder"] == "avhubert" for name in cfg["architectures"])
if needs_avhubert:
    subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "avhubert", "--root", str(ROOT)], check=True)
    worker_python = str(ROOT / ".venv-avhubert/bin/python")
    settings.AVH_PYTHON = worker_python
    cfg["encoders"]["avhubert"]["python"] = worker_python
    print("Worker ready:", worker_python)
else:
    print("Các model đã chọn chỉ dùng FATE: bỏ qua worker AV-HuBERT.")


## 3. Trích feature và train
Prepare dùng chung cache giữa các model cùng encoder. Nó có thể trích feature test nhưng không fit thống kê hoặc train trên test.
Nếu phiên bị ngắt, giữ cache/run và chạy lại với RESUME=True, cùng dữ liệu/cấu hình/code.


In [ ]:
execute("prepare", cfg)


In [ ]:
execute("train", cfg, resume=RESUME)  # Chỉ train, không compare và không tự evaluate.


## 4. Xem từng detector sau huấn luyện
Biểu đồ loss/validation theo epoch; ROC, PR, confusion matrix và phân bố score của best checkpoint.
Ảnh timeline dùng một real và một fake đầu tiên theo sample ID, không chọn theo độ đẹp của dự đoán.
Kết quả validation đã được dùng chọn checkpoint/ngưỡng, nên không coi là kết quả test độc lập.
Có thể chạy lại riêng cell này từ thư mục artifact đã tải xuống; chỉ cần JSON và thư viện vẽ, không cần GPU/video.


In [ ]:
from IPython.display import display, Markdown, Image
from vn_av_df.reporting import training_reports

def fmt(value):
    return f"{value:.4f}" if isinstance(value, float) else ("-" if value is None else str(value))

def show_detector_reports(split="validation"):
    # Mỗi vòng là một detector/seed, không tạo bảng so sánh nhiều detector.
    for item in training_reports(cfg["runs"], split=split):
        info = item["summary"]
        display(Markdown(f"### {info['architecture']} - seed {info['seed']} - {split}"))
        display(Markdown(f"Best epoch: **{info['best_epoch']}** / {info['epochs_completed']} epoch; "
                         f"video coverage: **{info['video_coverage']:.1%}**."))
        lines = ["| Chỉ số video | Giá trị |", "|---|---|"]
        for name, value in info["video_metrics"].items():
            if name != "confusion_matrix":
                lines.append(f"| {name} | {fmt(value)} |")
        display(Markdown("\n".join(lines)))
        # Ô 2×2: fake tính cùng real gốc; real/sham riêng chỉ có ý nghĩa ở false_alarm_rate.
        lines = ["| Ô | N | ROC-AUC | F1 | False alarm |", "|---|---|---|---|---|"]
        for cell, m in info["by_condition"].items():
            lines.append(f"| {cell} | {m['samples']} | {fmt(m['roc_auc'])} | {fmt(m['f1'])} | {fmt(m['false_alarm_rate'])} |")
        display(Markdown("\n".join(lines)))
        if info.get("branches"):
            # P2: AUC từng nhánh, real gốc (0) so với từng ô (1); dòng sham = độ nhạy lệch A/V.
            scores = info["branches"]["scores"]
            cells = sorted({c for v in scores.values() for c in v})
            lines = ["| Nhánh | " + " | ".join(cells) + " |", "|---" * (len(cells) + 1) + "|"]
            for name, values in scores.items():
                lines.append(f"| {name} | " + " | ".join(fmt(values.get(c, {}).get("roc_auc")) for c in cells) + " |")
            display(Markdown("\n".join(lines)))
        display(Markdown(f"Trainable parameters: **{info['resources']['trainable_parameters']:,}**; "
                         f"thời gian train trong phiên: **{info['resources']['elapsed_this_session_s']:.1f}s**."))
        if info.get("reconstruction"):
            display(Markdown(f"Stage A→V real-only - best validation loss: "
                             f"**{info['reconstruction']['best_validation_loss']:.4f}**; "
                             f"constant predictor: **{info['reconstruction']['constant_validation_loss']:.4f}**."))
        if info.get("sync"):
            display(Markdown(f"Stage S sync (không thấy fake) - best validation loss: "
                             f"**{info['sync']['best_validation_loss']:.4f}**; "
                             f"AUC real vs sham/dịch lệch: **{fmt(info['sync']['best_validation_auc'])}**."))
        for path in item["images"]:
            display(Image(filename=path))

show_detector_reports("validation")


## 5. Test tùy chọn
Mặc định bỏ qua. Khi RUN_TEST=True, đánh giá riêng từng detector với checkpoint/ngưỡng đã khóa trên validation.
Không tạo bảng xếp hạng hay paired comparison; chỉ lưu test.json, predictions-test.json và ảnh từng detector.


In [ ]:
if RUN_TEST:
    execute("evaluate", cfg)
    show_detector_reports("test")
else:
    print("Chưa chạy test. Báo cáo bên trên là validation.")


## 6. Tải output
ZIP mang tên RUN_NAME, chứa checkpoint, cấu hình, part/hash, dự đoán và hình báo cáo.
Giữ các ZIP này để đưa vào notebook compare.ipynb sau này; notebook đó hiện chỉ có khung.
Video/cache không được đóng vào ZIP kết quả để tránh file quá lớn.


In [ ]:
import zipfile
archive = Path("/kaggle/working") / f"{RUN_NAME}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (Path(cfg["runs"])).rglob("*"):
        if p.is_file():
            z.write(p, p.relative_to(ROOT))
    if needs_avhubert:
        for p in (ROOT / ".kaggle-tools/avhubert").glob("*"):
            if p.is_file():
                z.write(p, "environment/avhubert/" + p.name)
print("Tải file:", archive)
